# 氟化制冷剂吸收预测模型 （基于三端全局图网络 Tri-Graph GNN）
本 Notebook 用于在 Google Colab 上拉取最新代码，一键运行数据预处理、模型训练以及可解释性分析。

## 💡 流程说明：每次您本地修改代码并 `git push` 后：
1. 在 Colab 中**依次运行**下面的单元格。
2. 在 **第 2 步 (Git 同步)** 中，会自动检查是否有更新：如果是首次运行会自动 `clone`；如果是后续运行则会自动 `git pull`，确保您始终跑的是最新代码！

## 1. 安装 PyG (PyTorch Geometric) 及其依赖与化学库

In [ ]:
# 安装图神经网络所需的 PyG 依赖库以及 RDKit
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install rdkit pandas numpy matplotlib networkx tqdm openpyxl

## 2. 自动拉取或更新 GitHub 仓库代码

> **⚠️ 注意**：请确保您在 Colab 左侧的 **Secrets (钥匙图标)** 处添加了一个 Secret，其值为您的 GitHub Personal Access Token。

In [ ]:
import os
from google.colab import userdata

# 1. 确保工作在 /content 目录下
%cd /content

# 2. 尝试从 Colab Secrets 获取访问令牌 (兼容 GOOGL1 或 COCOLI)
token = None
for secret_key in ['GOOGL1', 'COCOLI']:
    try:
        token = userdata.get(secret_key)
        if token:
            print(f"🔑 已成功获取 Secret 键: '{secret_key}'")
            break
    except Exception:
        continue

if not token:
    print("⚠️ 警告: 未在 Colab Secrets 中找到 'GOOGL1' 或 'COCOLI'。如果是公开仓库可继续，私有仓库克隆可能会失败。")

repo_path = "/content/GNN"
repo_url = f"https://{token}@github.com/badaozhiwei-cmyk/GNN.git" if token else "https://github.com/badaozhiwei-cmyk/GNN.git"

if not os.path.exists(repo_path):
    # 首次克隆
    print("🚀 正在克隆 GNN 仓库...")
    !git clone {repo_url}
else:
    # 增量拉取最新修改，每次改动直接拉取即可！
    print("🔄 仓库已存在，正在拉取最新更改 (git pull)...")
    %cd {repo_path}
    !git pull origin main

# 最终切入项目根目录 (直接是 /content/GNN)
%cd {repo_path}

## 3. 生成 3-Graph 体系数据集
每次您修改了 `prepare_tri_graph_data.py` 中的特征（例如新增了电负性和共价半径），或者想重新处理数据，请运行此步骤。

In [ ]:
# 切换到 GNN 工作根目录并运行预处理
%cd /content/GNN
!python prepare_tri_graph_data.py

## 4. 运行模型训练 (以 GIN 算法为例)
自动读取生成的 `processed_tri_data` 进行溶解度预测回归训练，运行完毕后会生成对应的预测效果图。

In [ ]:
# 进入 GNN_for_property_prediction 训练目录并运行模型
%cd /content/GNN/GNN_for_property_prediction
!python GIN_Runner_v2.py

## 5. 运行可解释性分析 (GNNExplainer)
我们为您准备了三种级别的分析方式（请按需选用）：

In [ ]:
# 切换到可解释性分析文件夹
%cd /content/GNN/Explainer_for_ionic_molecule

# ========================================== 
# 【模式 1】：1分钟快速冒烟测试 (跑3个样本)
# ========================================== 
!python smoke_test_v2.py --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth

# ========================================== 
# 【模式 2】：指定样本生成单分子热力图 (推荐！)
# 运行完毕后，会在 fragment_explain_result 目录生成可视化 PNG 图片
# ========================================== 
# !python single_explain_v2.py --sample_idx 100 --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth

# ========================================== 
# 【模式 3】：大论文全量统计分析 (大约需要1小时)
# 运行完毕后，会自动保存元素重要性贡献图与 7 大节点特征重要性图
# ========================================== 
# !python fragment_explain_v2.py --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth